In [1]:
"""
add_coordinates.py
==================
Adds precise Tracker instrument coordinates from station_location_ids.csv
to all three SURFRAD preprocessed output files.

Tracker location used for each station — this is where the solar
tracking instruments (DNI pyrheliometer, shaded diffuse pyranometer)
are physically mounted.

Run this from your Windows machine:
  python add_coordinates.py
"""

import pandas as pd
from pathlib import Path

# ============================================================
# Exact Tracker coordinates from station_location_ids.csv
# Desert Rock had spaces in degree values — cleaned here
# ============================================================
TRACKER_COORDS = {
    "bon": {"lat":  40.05195, "lon": -88.37310},   # Bondville Tracker
    "fpk": {"lat":  48.30780, "lon": -105.10172},  # Fort Peck Tracker
    "gwn": {"lat":  34.25470, "lon": -89.87290},   # Goodwin Creek (single location)
    "tbl": {"lat":  40.12493, "lon": -105.23677},  # Table Mountain Tracker
    "dra": {"lat":  36.62387, "lon": -116.01948},  # Desert Rock Tracker
    "psu": {"lat":  40.72023, "lon": -77.93090},   # Penn State Tracker
    "sxf": {"lat":  43.73399, "lon": -96.62328},   # Sioux Falls Tracker
}

STATION_NAMES = {
    "bon": "Bondville, IL",
    "fpk": "Fort Peck, MT",
    "gwn": "Goodwin Creek, MS",
    "tbl": "Table Mountain, CO",
    "dra": "Desert Rock, NV",
    "psu": "Penn State, PA",
    "sxf": "Sioux Falls, SD",
}

# ---- Set your path here ----
BASE = Path(r"C:\Users\mgvhy\OneDrive - University of Missouri\scientific_data\analysis\surfrad_processed")

# ============================================================
# 1. surfrad_daily_clearsky.csv
# ============================================================
df1 = pd.read_csv(BASE / "surfrad_daily_clearsky.csv")

# Replace approximate coords with precise tracker coords
df1["lat"] = df1["station"].map({k: v["lat"] for k, v in TRACKER_COORDS.items()})
df1["lon"] = df1["station"].map({k: v["lon"] for k, v in TRACKER_COORDS.items()})

# Add full station name after station code column
df1.insert(2, "station_full", df1["station"].map(STATION_NAMES))

df1.to_csv(BASE / "surfrad_daily_clearsky.csv", index=False)
print(f"Updated: surfrad_daily_clearsky.csv ({len(df1)} records)")
print(df1[["station","station_full","lat","lon"]].drop_duplicates().to_string(index=False))

# ============================================================
# 2. surfrad_multiyear_means.csv
# ============================================================
df2 = pd.read_csv(BASE / "surfrad_multiyear_means.csv")

df2["lat"] = df2["station"].map({k: v["lat"] for k, v in TRACKER_COORDS.items()})
df2["lon"] = df2["station"].map({k: v["lon"] for k, v in TRACKER_COORDS.items()})
df2.insert(2, "station_full", df2["station"].map(STATION_NAMES))

df2.to_csv(BASE / "surfrad_multiyear_means.csv", index=False)
print(f"\nUpdated: surfrad_multiyear_means.csv ({len(df2)} rows)")

# ============================================================
# 3. surfrad_data_availability.csv — pivot table structure
# Add coordinates as a companion reference file instead
# ============================================================
coords_ref = pd.DataFrame([
    {
        "station"      : k,
        "station_full" : STATION_NAMES[k],
        "lat_tracker"  : v["lat"],
        "lon_tracker"  : v["lon"],
    }
    for k, v in TRACKER_COORDS.items()
])

coords_ref.to_csv(BASE / "station_coordinates.csv", index=False)
print(f"\nSaved: station_coordinates.csv")
print(coords_ref.to_string(index=False))

print("\nDone. All files updated with precise tracker coordinates.")

Updated: surfrad_daily_clearsky.csv (590 records)
station       station_full      lat        lon
    bon      Bondville, IL 40.05195  -88.37310
    dra    Desert Rock, NV 36.62387 -116.01948
    fpk      Fort Peck, MT 48.30780 -105.10172
    gwn  Goodwin Creek, MS 34.25470  -89.87290
    psu     Penn State, PA 40.72023  -77.93090
    sxf    Sioux Falls, SD 43.73399  -96.62328
    tbl Table Mountain, CO 40.12493 -105.23677

Updated: surfrad_multiyear_means.csv (84 rows)

Saved: station_coordinates.csv
station       station_full  lat_tracker  lon_tracker
    bon      Bondville, IL     40.05195    -88.37310
    fpk      Fort Peck, MT     48.30780   -105.10172
    gwn  Goodwin Creek, MS     34.25470    -89.87290
    tbl Table Mountain, CO     40.12493   -105.23677
    dra    Desert Rock, NV     36.62387   -116.01948
    psu     Penn State, PA     40.72023    -77.93090
    sxf    Sioux Falls, SD     43.73399    -96.62328

Done. All files updated with precise tracker coordinates.
